In [1]:
# Cài đặt các thư viện cần thiết cho Selenium trên Google Colab
!pip install selenium
!apt-get update
!apt-get install -y chromium-chromedriver

# Cấu hình để Colab nhận diện được chromedriver
import sys
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
from selenium import webdriver
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 32.6 MB/s eta 0:00:00
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,080 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,354 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 k

In [2]:
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

class FacebookScraper:
    def __init__(self, email, password):
        self.email = email
        self.password = password
        self.driver = None

    def initialize_driver(self):
        """Khởi tạo Chrome Driver với cấu hình đặc biệt cho Colab."""
        options = webdriver.ChromeOptions()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument("--window-size=1920,1080")

        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)
        options.add_argument("--disable-notifications")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

        self.driver = webdriver.Chrome(options=options)
        print("✅ Khởi tạo trình duyệt ảo thành công!")

    def simulate_human_typing(self, element, text):
        for char in text:
            element.send_keys(char)
            time.sleep(random.uniform(0.08, 0.25))

    def login(self):
        """Đăng nhập Facebook."""
        self.driver.get("https://www.facebook.com/login")
        email_input = WebDriverWait(self.driver, 15).until(EC.presence_of_element_located((By.NAME, "email")))
        self.simulate_human_typing(email_input, self.email)
        time.sleep(random.uniform(0.5, 1.5))

        password_input = WebDriverWait(self.driver, 15).until(EC.presence_of_element_located((By.NAME, "pass")))
        self.simulate_human_typing(password_input, self.password)
        time.sleep(random.uniform(0.5, 1.5))

        login_button = self.driver.find_element(By.XPATH, "//button[@type='submit']")
        login_button.click()
        print("🔑 Đang đăng nhập...")

        WebDriverWait(self.driver, 25).until(
            EC.presence_of_element_located((By.XPATH, "//div[@role='navigation']"))
        )
        print("✅ Đăng nhập thành công!")
        time.sleep(random.uniform(2, 4))

    def navigate_to_profile(self, profile_url):
        print(f"🌐 Đang truy cập: {profile_url}")
        self.driver.get(profile_url)
        time.sleep(random.uniform(4, 7))

    def slow_scroll(self, step=700):
        self.driver.execute_script(f"window.scrollBy(0, {step});")
        time.sleep(random.uniform(2.5, 5.5))

    def expand_all_see_more(self):
        """Mở rộng các đoạn 'Xem thêm' trong bài viết."""
        try:
            see_more_buttons = self.driver.find_elements(By.XPATH, "//*[text()='Xem thêm' or text()='See more']")
            for btn in see_more_buttons:
                try:
                    self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                    time.sleep(random.uniform(0.5, 1))
                    self.driver.execute_script("arguments[0].click();", btn)
                    time.sleep(random.uniform(1, 2.5))
                except Exception:
                    continue
        except Exception:
            pass

    def expand_all_comments(self, max_clicks=5):
        """Mở rộng tất cả bình luận bằng cách click 'Xem thêm bình luận'."""
        for _ in range(max_clicks):
            try:
                buttons = self.driver.find_elements(By.XPATH, "//div[contains(text(), 'Xem thêm bình luận') or contains(text(), 'View more comments')]")
                if not buttons:
                    break
                for btn in buttons:
                    try:
                        self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                        time.sleep(random.uniform(0.5, 1.2))
                        self.driver.execute_script("arguments[0].click();", btn)
                        time.sleep(random.uniform(1, 2))
                    except Exception:
                        continue
            except Exception:
                continue

    def extract_posts_with_bs(self):
        """Cào nội dung bài viết, lượt like, thời gian và bình luận (đã lọc trùng chính xác)."""
        self.expand_all_see_more()
        self.expand_all_comments()
        time.sleep(random.uniform(1.5, 3))

        soup = BeautifulSoup(self.driver.page_source, "html.parser")
        posts_data = []

        posts = soup.find_all("div", attrs={"aria-posinset": True})
        for post in posts:
            try:
                # ====== LẤY NỘI DUNG BÀI VIẾT ======
                message_elements = post.find_all("div", {"data-ad-preview": "message"})
                post_text = " ".join([msg.get_text(" ", strip=True) for msg in message_elements]) or None

                # ====== LẤY THỜI GIAN ======
                time_tag = post.find("a", href=lambda x: x and "posts" in x)
                post_time = time_tag.get_text(strip=True) if time_tag else "Không xác định"

                # ====== LẤY LƯỢT LIKE ======
                likes = "0"
                like_spans = post.find_all("span")
                for span in like_spans:
                    text = span.get_text(strip=True)
                    if any(k in text.lower() for k in ["thích", "likes", "reactions"]):
                        likes = text
                        break

                # ====== LẤY BÌNH LUẬN ======
                comments = []
                # Tìm khối chứa các bình luận thực sự
                comment_sections = post.find_all("div", attrs={"aria-label": lambda v: v and ("bình luận" in v.lower() or "comment" in v.lower())})
                for section in comment_sections:
                    for cmt in section.find_all("div", {"dir": "auto"}):
                        text = cmt.get_text(" ", strip=True)
                        if text and len(text.split()) > 3 and text != post_text:
                            comments.append(text)

                # Lọc trùng bình luận
                comments = list(dict.fromkeys(comments))[:5]

                # ====== LƯU DỮ LIỆU ======
                if post_text:
                    posts_data.append({
                        "post_text": post_text,
                        "post_time": post_time,
                        "likes": likes,
                        "comments": comments
                    })

            except Exception:
                continue

        return posts_data

    def remove_duplicates(self, data_list):
        seen = set()
        unique_data = []
        for data in data_list:
            identifier = data['post_text']
            if identifier not in seen:
                seen.add(identifier)
                unique_data.append(data)
        return unique_data

    def scrape_posts(self, max_posts):
        all_posts = []
        scroll_count = 0
        while len(all_posts) < max_posts:
            if scroll_count > 0 and scroll_count % random.randint(3, 4) == 0:
                rest_time = random.uniform(8, 15)
                print(f"😴 Nghỉ {rest_time:.1f} giây để tránh bị phát hiện...")
                time.sleep(rest_time)
            self.slow_scroll()
            scroll_count += 1
            posts = self.extract_posts_with_bs()
            all_posts.extend(posts)
            all_posts = self.remove_duplicates(all_posts)
            print(f"📦 Đã cào được {len(all_posts)} bài viết duy nhất...")
            if len(all_posts) >= max_posts:
                break
        return all_posts[:max_posts]

    def save_to_text_file(self, posts_data, filename="facebook_posts.txt"):
        with open(filename, "w", encoding="utf-8") as f:
            f.write("📘 NỘI DUNG CÁC BÀI POST GẦN NHẤT TỪ FANPAGE DUE CONFESSIONS OFFICIAL\n")
            f.write("=" * 80 + "\n\n")
            for idx, post in enumerate(posts_data, start=1):
                f.write(f"BÀI POST {idx}:\n")
                f.write(f"Thời gian: {post.get('post_time', 'Không xác định')}\n")
                f.write(f"Likes: {post.get('likes', 'Không xác định')}\n")
                f.write(f"Nội dung: {post.get('post_text', 'Không có nội dung')}\n")
                f.write("💬 Bình luận:\n")
                for cmt in post.get("comments", []):
                    f.write(f"  - {cmt}\n")
                f.write("-" * 50 + "\n\n")
        print(f"✅ Đã lưu thành công {len(posts_data)} bài viết vào file: {filename}")
        from google.colab import files
        files.download(filename)

    def close(self):
        if self.driver:
            self.driver.quit()
            print("🚪 Đã đóng trình duyệt ảo.")


# ==============================
# 🔰 CHẠY THỰC TẾ
# ==============================
if __name__ == "__main__":
    USER_EMAIL = "hoquocuong2027@gmail.com"  # ⚠️ Có thể đổi sang biến môi trường để bảo mật
    USER_PASSWORD = "no1thedie123"

    scraper = FacebookScraper(USER_EMAIL, USER_PASSWORD)
    try:
        scraper.initialize_driver()
        scraper.login()
        scraper.navigate_to_profile("https://www.facebook.com/dueconfessionsofficial")
        posts_data = scraper.scrape_posts(max_posts=7)
        if posts_data:
            scraper.save_to_text_file(posts_data)
        else:
            print("⚠️ Không tìm thấy bài viết nào. Có thể selector đã thay đổi.")
    except Exception as e:
        scraper.driver.save_screenshot('error_screenshot.png')
        print("❌ Lỗi, đã chụp ảnh màn hình: error_screenshot.png")
        from google.colab import files
        files.download('error_screenshot.png')
        print(f"Lỗi chi tiết: {e}")
    finally:
        scraper.close()

✅ Khởi tạo trình duyệt ảo thành công!
🔑 Đang đăng nhập...
✅ Đăng nhập thành công!
🌐 Đang truy cập: https://www.facebook.com/dueconfessionsofficial
📦 Đã cào được 6 bài viết duy nhất...
📦 Đã cào được 6 bài viết duy nhất...
📦 Đã cào được 6 bài viết duy nhất...
📦 Đã cào được 6 bài viết duy nhất...
📦 Đã cào được 8 bài viết duy nhất...
✅ Đã lưu thành công 7 bài viết vào file: facebook_posts.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚪 Đã đóng trình duyệt ảo.
